In [20]:
import torch
import numpy as np
from scipy.linalg import expm, solve_continuous_lyapunov
from scipy.integrate import solve_ivp

# ------------------- UTILITIES -------------------
def compute_covariance_ode(t, beta, A, G, Sigma_0):
    """Numerical covariance via ODE integration."""
    n = A.shape[0]
    F = -beta * A
    Q = G @ G.T

    def lyapunov_ode(_, sigma_flat):
        Sigma = sigma_flat.reshape(n, n)
        dSigma_dt = F @ Sigma + Sigma @ F.T + Q
        return dSigma_dt.flatten()

    sol = solve_ivp(lyapunov_ode, [0, t], Sigma_0.flatten(), t_eval=[t], method='BDF', rtol=1e-10, atol=1e-12)
    return sol.y[:, -1].reshape(n, n)

def compute_mean_and_covariance(t, beta, A, G, mu_0, Sigma_0):
    """Mean + covariance via ODE propagation."""
    F = -beta * A
    M_t = expm(F * t)
    mu_t = M_t @ mu_0
    Sigma_t = compute_covariance_ode(t, beta, A, G, Sigma_0)
    return mu_t, Sigma_t

def compute_covariance_analytical(t, beta, A, G, Sigma_0, C):
    """Analytical covariance using stationary solution C."""
    F = -beta * A
    M_t = expm(F * t)
    return C + M_t @ (Sigma_0 - C) @ M_t.T

def stationary_covariance(beta, A, G):
    """Solve F C + C F^T + Q = 0 for C."""
    F = -beta * A
    Q = G @ G.T
    return solve_continuous_lyapunov(F, -Q)

# ------------------- SETUP -------------------
gamma = 1.0
lambda_val = 2.0
M = 0.5
c = 0.1
beta = 1.0
t = 0.5

A = torch.tensor([
    [0., -1/M, 0.],
    [1, gamma**2/M, gamma * lambda_val * c],
    [0., gamma * lambda_val * c, lambda_val**2]
], dtype=torch.float64)

eigs = np.linalg.eigvals(A.numpy())
print("Eigenvalues of A:", eigs)
print("All real parts positive?", np.all(np.real(eigs) > 0))

B = torch.tensor([
    [0., 0., 0.],
    [0., gamma, 0.],
    [0., lambda_val * c, lambda_val * torch.sqrt(torch.tensor(1 - c**2, dtype=torch.float64))]
], dtype=torch.float64)

A_np, B_np = A.numpy(), B.numpy()
G = np.sqrt(2*beta) * B_np

# ------------------- LYAPUNOV VERIFICATION -------------------
C_np = stationary_covariance(beta, A_np, G)
LHS = (-beta * A_np) @ C_np + C_np @ (-beta * A_np).T + G @ G.T
print("Stationary covariance residual (should be ~0):\n", LHS)
print("C is:\n", C_np)

# ------------------- TEST: ODE VS ANALYTICAL -------------------
mu0 = np.zeros(3)
Sigma0 = 2 * np.eye(3)

_, Sigma_num = compute_mean_and_covariance(t, beta, A_np, G, mu0, Sigma0)
Sigma_ana = compute_covariance_analytical(t, beta, A_np, G, Sigma0, C_np)

print("\nNumerical Σ(t):\n", Sigma_num)
print("Analytical Σ(t):\n", Sigma_ana)
print("Close?", np.allclose(Sigma_num, Sigma_ana, atol=1e-6))


Eigenvalues of A: [0.99204433+1.00395413j 0.99204433-1.00395413j 4.01591134+0.j        ]
All real parts positive? True
Stationary covariance residual (should be ~0):
 [[ 1.12798454e-15  3.33066907e-16 -9.36750677e-17]
 [ 1.11022302e-16 -1.33226763e-15  5.55111512e-17]
 [-5.55111512e-17  5.55111512e-17  8.88178420e-16]]
C is:
 [[9.95363215e-01 2.29836071e-16 7.72797527e-03]
 [3.34156200e-16 4.98454405e-01 1.54559505e-02]
 [7.72797527e-03 1.54559505e-02 9.99227202e-01]]

Numerical Σ(t):
 [[ 2.18565953 -0.02779133 -0.00672007]
 [-0.02779133  0.6724903   0.00236883]
 [-0.00672007  0.00236883  1.01856995]]
Analytical Σ(t):
 [[ 2.18565953 -0.02779133 -0.00672007]
 [-0.02779133  0.6724903   0.00236883]
 [-0.00672007  0.00236883  1.01856995]]
Close? True
